<a href="https://colab.research.google.com/github/LSDtopotools/lsdtt_notebooks/blob/master/basic_examples/junction_angles_and_flow_directions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Calculate junction angles and flow directions for a channel network

This notebook uses LSDTopoTools to calculate tributary junction angles and flow directions for a channel network.

For the junction angles methodology, you can refer to this paper: https://www.pnas.org/doi/abs/10.1073/pnas.2211942119

To claculate flow directions, the code extracts channel segments - i.e. the whole channel profile in between each river junction. It then calculates the bearing of each channel segment.

You must upload your DEM in TIFF format and in a WGS84 UTM coordinate system.

Authors: Fiona Clubb, based on original notebook/code from Simon Mudd

## Get the right python packages

In this first cell we're going to install `lsdtopotools` and `lsdviztools`.
**WARNING - the first time you do this, it might take a few minutes. Just wait for it to finish and then you don't have to run it again for the rest of the notebook.**



First we install the gdal command line tools, which we use later. These have a bunch of useful ways to maniupulate topographic data.

In [1]:
!apt-get install gdal-bin &> /dev/null

Now we need to install lsdtopotools. We get the package from a server at the University of Edinburgh and then install.

In [2]:
# download lsdtt-basic-metrics and make available
!apt-get install libfftw3-dev &> /dev/null
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_dev_ubuntu/lsdtt-basic-metrics
!mv lsdtt-basic-metrics /usr/local/sbin/
!chmod 777 /usr/local/sbin/lsdtt-basic-metrics

--2026-01-28 10:33:50--  https://www.geos.ed.ac.uk/~smudd/lsdtt_dev_ubuntu/lsdtt-basic-metrics
Resolving www.geos.ed.ac.uk (www.geos.ed.ac.uk)... 129.215.8.163
Connecting to www.geos.ed.ac.uk (www.geos.ed.ac.uk)|129.215.8.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4386088 (4.2M)
Saving to: ‘lsdtt-basic-metrics’

lsdtt-basic-metrics 100%[===================>]   4.18M  5.37MB/s    in 0.8s    

2026-01-28 10:33:52 (5.37 MB/s) - ‘lsdtt-basic-metrics’ saved [4386088/4386088]



This is to install the python wrapper that makes topographic analysis routines python-friendly.  

In [3]:
!pip install lsdviztools &> /dev/null

The next line tests to see if it worked. If you get some output asking for a parameter file then lsdtopotools is installed. This notebook was tested on version 0.9.

In [4]:
!lsdtt-basic-metrics -v

|| Welcome to the LSDTopoTools basic metrics tool!     ||
|| This program has a number of options for calculating||
|| simple landscape metrics.                           ||
|| This program was developed by Simon M. Mudd         ||
||  at the University of Edinburgh                     ||
|| If you use these routines please cite:              ||
|| http://doi.org/10.5281/zenodo.4577879               ||
|| If you use the roughness routine please cite:       ||
|| https://www.doi.org/10.5194/esurf-3-483-2015        ||
|| Documentation can be found at:                      ||
|| https://lsdtopotools.github.io/LSDTT_documentation/ ||
|| This is LSDTopoTools2 version                       ||
|| 0.10d
|| If the version number has a d at the end it is a    ||
||  development version.                               ||
|| You have called an LSDTopoTools program.            ||
|| Prepare to explore topographic data!                ||
|| You can find some examples of usage here:           ||
|| ht

## Import the packages that we need to the notebook

First check the version of lsdviztools. **For this notebook we need lsdviztools >=0.4.9**

In [5]:
import lsdviztools
lsdviztools.__version__

'0.4.15'

You may get an "Import Error" with the next step. If you do, try clicking on "Runtime" in the top menu bar, then "Restart session", and then try running the cell below again.

In [6]:
import lsdviztools.lsdmapwrappers as lsdmw
import lsdviztools.lsdbasemaptools as bmt
from lsdviztools.lsdplottingtools import lsdmap_gdalio as gio

#pandas
import pandas as pd
import geopandas as gpd

#colab
from google.colab import files

#general
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm
%matplotlib inline

## Upload your DEM to Google Colab

The next step is to upload your DEM for your area of interest to Google Colab. To do this, click on the small folder icon on the left hand bar and then click `Upload to Session Storage`.

Make sure you wait until the DEM is fully uploaded before running the cell below, or you will get an error!

Your DEM should be in GeoTiff format and **needs to be in a UTM WGS84 coordinate system.**

LSDTopoTools requires data in a slightly different format (ENVI bil). The following line uses gdal to convert your DEM into the ENVI bil format.

--------

**NOTE: Make sure you modify the variables "DEM_name" and "res" below to match the name of your GeoTiff file and the resolution of your DEM. In my example, I have uploaded a DEM called "Hokitika_15m.tif" which has a 15 m grid resolution.**

________

In [7]:
DEM_name = 'Hokitika_15m'       # name of your DEM without the .tif extension
res = 15                        # Your DEM resolution

!gdalwarp -of ENVI -r cubic -dstnodata -9999 -tr {res} {res} {DEM_name}.tif {DEM_name}.bil

Creating output file that is 1990P x 2448L.
Processing Hokitika_15m.tif [1/1] : 0Using internal nodata values (e.g. -9999) for image Hokitika_15m.tif.
...10...20...30...40...50...60...70...80...90...100 - done.


## Run the channel segment bearing code

Now we can go ahead and run the channel segment bearing analysis! LSDTopoTools is run through the command line. You can decide what analyses you want LSDTopoTools to do by giving it a parameter file - you just need to set the various options in the parameter file accordingly.

We can create a parameter file in Python by creating a dictionary with the parameters and values, and then writing this to a text file which will be read by LSDTopoTools.

In [8]:
# create the parameter dictionary
param_dict = {
    'read fname': DEM_name,                                 # the name of your DEM (declared above)
    'write fname': DEM_name,                                # code will write some other rasters - they will have a prefix matching your DEM name
    'write_hillshade': 'true',                              # set to true if you want to print a hillshade of the DEM
    'remove_seas': 'true',                                  # replace elevations <=0 with nodata
    'threshold_contributing_pixels': 1000,                  # drainage area threshold for channel extraction (in number of pixels)
    'print_channels_to_csv': 'true',                        # set to true if you want to print out a CSV file of the channel network
    'print_segment_bearings_and_gradients_to_csv': 'true',  # flag to turn on the segment bearing printing
    'convert_csv_to_geojson': 'true'                        # if true, convert the csv file to a geojson file (equivalent to shapefile) that can be easily loaded in GIS
    }

In [9]:
# write it to a text file
strings = [ f'{key}: {param_dict[key]}' for key in param_dict]
with open('LSDTT_channel_segments.param', 'w') as file:
  [file.write(f'{st}\n') for st in strings]

Now we have our parameter file, we can run the channel segment analysis:

In [10]:
!lsdtt-basic-metrics LSDTT_channel_segments.param

|| Welcome to the LSDTopoTools basic metrics tool!     ||
|| This program has a number of options for calculating||
|| simple landscape metrics.                           ||
|| This program was developed by Simon M. Mudd         ||
||  at the University of Edinburgh                     ||
|| If you use these routines please cite:              ||
|| http://doi.org/10.5281/zenodo.4577879               ||
|| If you use the roughness routine please cite:       ||
|| https://www.doi.org/10.5194/esurf-3-483-2015        ||
|| Documentation can be found at:                      ||
|| https://lsdtopotools.github.io/LSDTT_documentation/ ||
|| This is LSDTopoTools2 version                       ||
|| 0.10d
|| If the version number has a d at the end it is a    ||
||  development version.                               ||
|| You have called an LSDTopoTools program.            ||
|| Prepare to explore topographic data!                ||
|| You can find some examples of usage here:           ||
|| ht

We have now run our channel segment analysis! If you look in the files list, you should see many new files have been produced. You can download these and load them into a GIS.

The key files you want to download are:

* **DEM_name_CN.geojson**: geojson of the extracted channel network
* **DEM_name_segment_bearings.geojson** or .csv: files with the channel segment bearings. If you look in the attribute table, you will find a lot of different columns. The flow direction bearings are in the column `segment_bearing_whole_segment`. This is a point file where the point is located at the top of the channel segment where the bearing was calculated from.
* **DEM_name_segment_bearings_vector.geojson**: this is a polyline file which shows the channel segments as straight lines connecting each channel segment along the calculated bearing.

You can also look at some of these files in python directly:

In [13]:
# look at the first 5 lines of the segment bearings file
df = pd.read_csv("{}_segment_bearings.csv".format(DEM_name))
df.head()

,latitude,longitude,junction_number,junction_stream_order,junction_drainage_area,segment_bearing_whole_segment,segment_bearing_vertical_interval,gradient_whole_segment,gradient_vertical_interval,x_j,...,z_j,x_pu,y_pu,z_pu,x_vi,y_pi,z_vi,FlowDistance_junction,FlowDistance_pu,FlowDistance_vi
0,-42.910369,171.001729,0,1,225675,70.144783,70.144783,0.000050,0.000050,500141.156,...,39.039238,500681.156,5249333.5,39.001061,500681.156,500681.156,39.001061,772.461914,15.000000,15.000000
1,-42.911044,170.991990,2,1,239175,314.999969,314.999969,0.000050,0.000050,499346.156,...,39.032875,498971.156,5249438.5,39.006363,498971.156,498971.156,39.006363,657.609131,127.279221,127.279221
2,-42.907532,170.987213,3,2,3650625,314.999969,314.999969,0.000050,0.000050,498956.156,...,39.005302,498896.156,5249513.5,39.001061,498896.156,498896.156,39.001061,106.066017,21.213203,21.213203
3,-42.911450,170.995665,5,1,258975,308.990967,308.990967,0.000050,0.000050,499646.156,...,39.045601,499016.156,5249528.5,39.001061,499016.156,499016.156,39.001061,905.954285,15.000000,15.000000
4,-42.915097,171.005956,7,1,264375,39.805572,39.805572,0.000053,0.000053,500486.156,...,40.006363,500561.156,5248703.5,40.000000,500561.156,500561.156,40.000000,675.182739,554.116760,554.116760


In [15]:
# look at the first 5 lines of the channel network file
chan_df = pd.read_csv("{}_CN.csv".format(DEM_name))
chan_df.head()

,latitude,longitude,Junction Index,Stream Order,NI,receiver_NI,receiver_JI
0,-42.910369,171.00173,0,1,6483,6212,1
1,-42.910234,171.00191,0,1,6212,5942,1
2,-42.910099,171.00210,0,1,5942,5674,1
3,-42.909964,171.00228,0,1,5674,5944,1
4,-42.910099,171.00246,0,1,5944,6216,1


## Junction angles

The previous section extracted flow direction segments; this section extracts junction angles. It follows the same workflow: first we create a parameter file, then run lsdtopotools.

In [16]:
# create the parameter dictionary
param_dict = {
    'read fname': DEM_name,                                 # the name of your DEM (declared above)
    'write fname': DEM_name,                                # code will write some other rasters - they will have a prefix matching your DEM name
    'write_hillshade': 'false',                              # set to true if you want to print a hillshade of the DEM
    'remove_seas': 'true',                                  # replace elevations <=0 with nodata
    'threshold_contributing_pixels': 1000,                  # drainage area threshold for channel extraction (in number of pixels)
    'print_channels_to_csv': 'true',                        # set to true if you want to print out a CSV file of the channel network
    'print_junction_angles_to_csv': 'true',                 # flag to turn on junction angle printing
    'convert_csv_to_geojson': 'true'                        # if true, convert the csv file to a geojson file (equivalent to shapefile) that can be easily loaded in GIS
    }

In [17]:
# write it to a text file
strings = [ f'{key}: {param_dict[key]}' for key in param_dict]
with open('LSDTT_junction_angles.param', 'w') as file:
  [file.write(f'{st}\n') for st in strings]

Now we have our parameter file, we can run the channel segment analysis:

In [18]:
!lsdtt-basic-metrics LSDTT_junction_angles.param

|| Welcome to the LSDTopoTools basic metrics tool!     ||
|| This program has a number of options for calculating||
|| simple landscape metrics.                           ||
|| This program was developed by Simon M. Mudd         ||
||  at the University of Edinburgh                     ||
|| If you use these routines please cite:              ||
|| http://doi.org/10.5281/zenodo.4577879               ||
|| If you use the roughness routine please cite:       ||
|| https://www.doi.org/10.5194/esurf-3-483-2015        ||
|| Documentation can be found at:                      ||
|| https://lsdtopotools.github.io/LSDTT_documentation/ ||
|| This is LSDTopoTools2 version                       ||
|| 0.10d
|| If the version number has a d at the end it is a    ||
||  development version.                               ||
|| You have called an LSDTopoTools program.            ||
|| Prepare to explore topographic data!                ||
|| You can find some examples of usage here:           ||
|| ht

You should now have two additional files:

* DEM_Name_FULL_JAngles.csv and .geojson. These are point files with the junction angle data. If we look at the attribute table you'll see there are a lot of columns, the angles are stored in the column `donors_junction_angle`. Note that this code does not calculate angles for small stream segments (they must have at least 3 channel pixels). It also does not calculate angles where 3 tributaries join.